In [1]:
try:
    precio = int("hola")
    print("Precio:", precio)
except ValueError:
    print("ValueError: no pude convertir 'gratis' a entero")

ValueError: no pude convertir 'gratis' a entero


### 1.2) `ZeroDivisionError` — división entre cero

In [2]:
try:
    media = 100 / 0
    print("Media:", media)
except ZeroDivisionError:
    print("ZeroDivisionError: no se puede dividir entre cero")

ZeroDivisionError: no se puede dividir entre cero


### 1.3) `KeyError` — clave que no existe en un diccionario

Asumes que viene `"price"` pero el campo tiene otro nombre o falta.

In [3]:
post = {"id": 1, "title": "Título del post", "body": "Cuerpo del post", "userId": 1}

# Clave que no existe
try:
    print("Categoría:", post["category"])
except KeyError:
    print("KeyError: no hay clave 'category'")

KeyError: no hay clave 'category'


### 1.5) Varios `except` en el mismo `try`

Puedes capturar **más de un tipo** de error en la misma operación:

In [4]:
def mostrar_user_id(post: dict) -> None:
    """Intenta leer userId del post (como en JSONPlaceholder)."""
    try:
        user_id = int(post["userId"])  # KeyError si falta, ValueError si no es número
        print("userId:", user_id)
    except KeyError:
        print("Falta la clave 'userId'")
    except ValueError:
        print("userId no es un número válido")


mostrar_user_id({"id": 1, "title": "Hola", "userId": "3"})
mostrar_user_id({"id": 2, "title": "Sin user"})  # sin userId
mostrar_user_id({"id": 3, "title": "Mal", "userId": "abc"})

userId: 3
Falta la clave 'userId'
userId no es un número válido


### 1.6) `except Exception` — captura genérica

Si no sabes qué error concreto va a salir, puedes usar al final:

```python
except Exception as e:
    print("Exception:", e)
```

In [5]:
def probar(opcion):
    print(f"--- opción {opcion} ---")
    try:
        if opcion == 1:
            print(10 / 0)  # ZeroDivisionError → Exception
        elif opcion == 2:
            int("xyz")  # ValueError
        else:
            print("todo bien!")
    except ValueError:
        print("Capturado: ValueError")
    except Exception as e:
        print("Capturado: Exception →", e)


probar(1)  # verás Exception
probar(2)  # verás ValueError
probar(33)  # verás "todo bien"

--- opción 1 ---
Capturado: Exception → division by zero
--- opción 2 ---
Capturado: ValueError
--- opción 33 ---
todo bien!


### 1.7) Bloque `else` — solo si **no** hubo error

Opcional y sencillo: el `else` se ejecuta cuando el `try` terminó **sin** excepción.

In [6]:
texto_id = input("introduce ID")

try:
    product_id = int(texto_id)
except ValueError:
    print("ID inválido")
else:
    print("ID válido, podemos llamar a la API con:", product_id)

ID inválido


### 1.8) Bloque `finally` — se ejecuta **siempre**

`finally` corre **al terminar** el `try` / `except` / `else`, haya error o no.

In [7]:
def procesar_id(texto_id: str) -> None:
    print(f"--- Procesando '{texto_id}' ---")
    try:
        product_id = int(texto_id)
        print("ID convertido:", product_id)
    except ValueError:
        print("No se pudo convertir el ID")
    else:
        print("Listo para llamar a la API con id =", product_id)
    finally:
        print("Fin del intento (finally siempre se ejecuta)\n")


procesar_id("3")      # caso OK
procesar_id("abc")    # caso con ValueError

--- Procesando '3' ---
ID convertido: 3
Listo para llamar a la API con id = 3
Fin del intento (finally siempre se ejecuta)

--- Procesando 'abc' ---
No se pudo convertir el ID
Fin del intento (finally siempre se ejecuta)



### 1.9) Lanzar excepciones con `raise`

Hasta ahora **capturábamos** errores con `except`. También puedes **generar** un error a propósito con **`raise`** cuando una condición no se cumple (dato inválido, regla de negocio, etc.).

```python
if condicion_mala:
    raise ValueError("Mensaje que describe el problema")
```

El programa se detiene en esa línea (si nadie captura el error) y verás el tipo y el mensaje, por ejemplo: `ValueError: La edad no puede ser negativa.`

In [9]:
# Prueba a introducir una edad negativa y otra positiva
edad = int(input("Ingrese su edad: "))


if edad < 0:
    raise ValueError("La edad no puede ser negativa.")

print("Edad válida:", edad)

ValueError: invalid literal for int() with base 10: 'werqwr'

Si **capturas** el error que tú mismo lanzaste, el programa no se cae:

**Salida esperada** (con `edad = -5`):

```text
Capturado: La edad no puede ser negativa.
```

A continuación, bloque completo con **try/except** y **raise**. Cuando se ejecuta **raise**, el siguiente bloque a ejecutar es el de **except**

In [10]:
try:
    edad = int(input("Ingrese su edad: "))
    if edad < 0:
        raise ValueError("La edad no puede ser negativa.")
    print("Edad válida:", edad)
except ValueError as e:
    print("Capturado:", e)

Capturado: invalid literal for int() with base 10: 'werq'


---
## 2) Manejo de errores llamando a una API REST

Pasos habituales:
1. Petición con **`timeout`** (tiempo máximo de espera).
2. Mirar **`status_code`**.
3. Si todo va bien, **`resp.json()`**.

In [ ]:
# %pip install requests -q

In [11]:
import requests

## 3) Códigos de estado HTTP (`status_code`)

| Rango | Significado | Ejemplos |
|-------|-------------|----------|
| **2xx** | Éxito | `200 OK`, `201 Created` |
| **4xx** | Error del **cliente** | `404 Not Found`, `400 Bad Request` |
| **5xx** | Error del **servidor** | `500 Internal Server Error` |

`requests` **no lanza excepción** solo por un 404: devuelve `resp` y tú miras `resp.status_code`.

Para convertir 4xx/5xx en excepción: **`resp.raise_for_status()`**.

### 3.1) Respuesta correcta (`200`)

 - API: **[JSONPlaceholder](https://jsonplaceholder.typicode.com/)**

In [12]:
resp = requests.get("https://jsonplaceholder.typicode.com/posts/1", timeout=10)
print("status:", resp.status_code)

if resp.ok:
    print("Petición OK — podemos leer JSON")
else:
    print("Revisar status antes de confiar en .json()")

status: 200
Petición OK — podemos leer JSON


### 3.2) Recurso no encontrado (`404`)

 - API: **[JSONPlaceholder](https://jsonplaceholder.typicode.com/)**

JSONPlaceholder devuelve **404** si el post no existe

In [13]:
url = "https://jsonplaceholder.typicode.com/posts/999999"
resp = requests.get(url, timeout=10)
print("status_code:", resp.status_code)
print("body[:200]:", resp.text[:200])

if resp.status_code == 404:
    print("Post no encontrado")

status_code: 404
body[:200]: {}
Post no encontrado


### 3.3) `raise_for_status()` con un 404

 - API: **[JSONPlaceholder](https://jsonplaceholder.typicode.com/)**
 
con **raise_for_status()**, forzamos que se lance una excepción, que luego capturamos en el bloque **except**

In [14]:
try:
    resp = requests.get("https://jsonplaceholder.typicode.com/posts/999999", timeout=10)
    print("status antes de raise:", resp.status_code)
    resp.raise_for_status()
    data = resp.json()
except requests.exceptions.HTTPError as e:
    print("HTTPError capturado:", e)

status antes de raise: 404
HTTPError capturado: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/999999


### 3.4) Error de servidor (`500`) 

 - API: **[HTTPbin](https://httpbin.org)**

JSONPlaceholder no devuelve 500 a propósito. Para ver un **5xx** usamos httpbin:

- `GET https://httpbin.org/status/500`

In [15]:
try:
    resp = requests.get("https://httpbin.org/status/500", timeout=10)
    print("status:", resp.status_code)
    resp.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("Error 5xx del servidor:", e)

status: 500
Error 5xx del servidor: 500 Server Error: INTERNAL SERVER ERROR for url: https://httpbin.org/status/500


---
## 6) Timeouts

 - API: **[HTTPbin](https://httpbin.org)**

Simulamos lentitud con httpbin (`/delay/3`) y `timeout=0.5`:

In [16]:
try:
    resp = requests.get("https://httpbin.org/delay/3", timeout=0.5)
    resp.raise_for_status()
except requests.exceptions.Timeout:
    print("Timeout: la API tardó más de lo permitido")

Timeout: la API tardó más de lo permitido


### 6.1) Timeout razonable con JSONPlaceholder (debe funcionar)

In [17]:
try:
    resp = requests.get("https://jsonplaceholder.typicode.com/posts/1", timeout=2)
    resp.raise_for_status()
    print("OK:", resp.json()["title"][:40], "...")
except requests.exceptions.Timeout:
    print("Timeout inesperado")

OK: sunt aut facere repellat provident occae ...


---
## 7) Errores de conexión (red)

Dominio inexistente o sin red → **`ConnectionError`** (no hay respuesta HTTP).

In [ ]:
try:
    requests.get("https://dominio-que-no-existe-xyz.example", timeout=5)
except requests.exceptions.ConnectionError as e:
    print("ConnectionError:", str(e)[:180], "...")

---
## 8) Función robusta: `obtener_post`

Patrón recomendado:

1. `timeout` en la petición.
2. `raise_for_status()`.
3. `try` / `except` por tipo de fallo.
4. Validar JSON.

In [ ]:
def obtener_post(post_id: int, timeout: int = 10):
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    print(url)
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return data
    except requests.exceptions.Timeout:
        print(f"Timeout al pedir post {post_id}")
    except requests.exceptions.HTTPError:
        print(f"HTTP {resp.status_code} al pedir post {post_id}")
    except ValueError:
        print("La respuesta no era JSON válido")
    except requests.exceptions.RequestException as e:
        print("Error de red:", e)
    return None


p = obtener_post(1)
if p:
    print("Éxito:", p["title"][:50], "...")

print("---")
obtener_post(999999)  # 404 esperado. El ID 999999 no existe